# **ZeptoFresh — 15-Minute Food & Essentials Delivery**
## Case Study · Week 05 · Day 27

# **Part (a) — Data Quality Diagnosis**

## **Issue 1 — delivery_time_mins = 0 (214 rows)**

**Classification:** Data entry error / structural issue

**Treatment:** A delivery time of zero is physically impossible — an order cannot be placed and received simultaneously. These 214 rows must be treated as invalid, not as genuine fast deliveries. Drop these rows entirely from analysis and model training. If the pipeline that generated them is known (e.g., cancelled orders marked as delivered), add a filter at ingestion.

## **Issue 2 — order_value_Rs = ₹2,95,000 (single bakery order)**

**Classification:** Outlier (extreme univariate outlier)

**Treatment:** The median order value is ₹310 and the mean is ₹620 — a ₹2.95 lakh bakery order is ~476× the median. Flag it using the IQR method: compute `Q3 + 3×IQR` as the upper fence and cap or remove values above it. Do not use the mean for this column in any downstream imputation — use the median instead. Verify with the ops team whether this was a bulk institutional order or a data entry error (e.g., ₹295 typed as ₹2,95,000).

## **Issue 3 — prep_time_mins has negative values**

**Classification:** Data entry error (impossible value)

**Treatment:** Preparation time cannot be negative — it is physically impossible to finish prep before starting it. Apply a hard validity filter: `df = df[df['prep_time_mins'] >= 0]`. If the source system is known to record timestamps incorrectly (e.g., end-time logged before start-time due to a clock sync bug), report the bug upstream and replace with the absolute value as a temporary fix only if the magnitude is plausible.

## **Issue 4 — customer_rating: 9,800 nulls + values of 0**

**Classification:** Missing values + invalid values (0 is outside the 1–5 scale)

**Treatment:** Two separate sub-problems. For the 9,800 nulls — ratings are likely missing because some customers never submitted a rating (MCAR). Do not impute with mean/median; instead, add a binary flag `rating_was_given` (0/1) and leave the nulls for models that handle missing input natively (e.g., XGBoost). For values recorded as 0 — these are invalid on a 1–5 scale; replace with `NaN` and apply the same missing-value treatment: `df['customer_rating'] = df['customer_rating'].replace(0, np.nan)`.

# **Part (b) — Distribution Analysis**

## **What does Mean > Median indicate?**

When **mean (18.4) > median (14.2)**, the distribution is **right-skewed (positive skew)**. The bulk of deliveries complete quickly (around 12–14 minutes), but a long tail of very late deliveries (up to 142 minutes) pulls the mean upward. The median is a more accurate representation of the typical delivery experience.

## **ASCII Histogram of delivery_time_mins**

In [ ]:

histogram = """
Frequency
  |
  |  ████
  | ██████
  |████████
  |██████████
  |████████████
  |██████████████
  |████████████████
  |██████████████████▌
  |████████████████████▌
  |██████████████████████▌  ▌
  |████████████████████████▌  ▌
  |__________________________________________→ delivery_time_mins
   0   5  10  14  18  25  35  50  80  142
              ↑          ↑
           median       mean
           (14.2)      (18.4)
"""
print(histogram)



Frequency
  |
  |  ████
  | ██████
  |████████
  |██████████
  |████████████
  |██████████████
  |████████████████
  |██████████████████▌
  |████████████████████▌
  |██████████████████████▌  ▌
  |████████████████████████▌  ▌
  |__________________________________________→ delivery_time_mins
   0   5  10  14  18  25  35  50  80  142
              ↑          ↑
           median       mean
           (14.2)      (18.4)



## **Transformation Before Modeling**

Apply a **log transformation**: `log1p(delivery_time_mins)`.

**Why:** Right-skewed features violate the normality assumption of linear models and cause tree-based models to make wasteful splits deep in the tail. `log1p` compresses the long right tail, making the distribution more symmetric and reducing the influence of extreme values like the 142-minute outlier. `log1p` (log(1+x)) is used instead of `log` to safely handle any zeros if they survive cleaning.

In [2]:
import numpy as np
import pandas as pd

# simulate the delivery_time distribution described in the case
np.random.seed(42)
delivery_times = np.concatenate([
    np.random.exponential(scale=10, size=9000) + 8,   # fast deliveries
    np.random.exponential(scale=30, size=1000) + 25   # late tail
])
delivery_times = np.clip(delivery_times, 1, 142)

original = pd.Series(delivery_times)
transformed = np.log1p(original)

print(f"Original  — mean: {original.mean():.2f}  median: {original.median():.2f}  skew: {original.skew():.2f}")
print(f"log1p     — mean: {transformed.mean():.2f}  median: {transformed.median():.2f}  skew: {transformed.skew():.2f}")


Original  — mean: 18.72  median: 14.81  skew: 2.43
log1p     — mean:  2.89  median:  2.77  skew: 0.61


# **Part (c) — Correlation Interpretation**

## **What is logically incorrect about the PM's conclusion?**

The PM commits the classic **correlation ≠ causation** error. The statement *"late deliveries cause refunds — so solving delay will eliminate refunds"* makes two unjustified logical leaps:

1. **Direction of causation is assumed, not proven.** r = +0.74 tells us the two variables move together, not which one drives the other. Refunds could also be causing delay measurements to be recorded differently (e.g., ops teams manually logging re-delivery times).

2. **Elimination is assumed from reduction.** Even if delay does cause some refunds, a correlation of 0.74 means 45% of the variance in refunds (`r² = 0.55`) is explained by delivery time — meaning **45% of refund variance comes from other factors entirely**. Fixing delay alone cannot eliminate refunds.

## **What does correlation actually indicate?**

Pearson's r = +0.74 indicates a **strong positive linear association**: as `delivery_time_mins` increases, `refund_issued` tends to increase with it. It quantifies the strength and direction of the relationship — nothing more. It makes no claim about:
- Which variable comes first in time
- Whether the relationship is causal
- Whether eliminating one will reduce the other

## **Two Possible Confounders**

**1. rain_flag** — Rain increases `delivery_time_mins` (r = +0.48 confirmed in the data) AND independently increases customer frustration, making them more likely to claim a refund regardless of actual delivery time. Rain is a common cause of both variables.

**2. order_category (Fresh Food / Bakery)** — Perishable items are both harder to deliver on time (more prep complexity) and more likely to arrive in poor condition, triggering refunds for quality reasons unrelated to delivery speed. A cold noodle bowl at 28 minutes earns a refund; a dry grocery packet at 28 minutes does not.

# **Part (d) — Bimodal Pattern in Tier-1 Cities**

## **Operational Reasons for the Bimodal Distribution**

**Peak 1 (12–14 minutes):** Orders fulfilled from the nearest micro-hub with available riders — standard operating conditions, hub is stocked, rider dispatched immediately.

**Peak 2 (28–32 minutes):** Two plausible operational explanations:
- **Hub overflow / rider unavailability** — during peak hours (lunch, dinner rush), the nearest hub has no free riders, so the order is reassigned to a second hub further away, adding ~15 minutes of extra rider travel
- **Out-of-stock rerouting** — certain SKUs are unavailable at the nearest hub and the order is partially or fully fulfilled from a backup hub, adding a transfer leg

The fact that this bimodality only appears in **Tier-1 cities** (Mumbai, Bangalore) is consistent with both explanations — these cities have higher order density, more frequent hub congestion, and more complex multi-hub logistics.

## **Why This Must Be Addressed Before Modeling**

A bimodal distribution violates the implicit assumption of most ML models that the feature comes from a single underlying process. The two peaks represent **two structurally different delivery regimes** — near-hub standard delivery vs. rerouted multi-hub delivery. If treated as one distribution:
- The model cannot learn the correct decision boundary for late-delivery risk
- Features that predict lateness in regime 1 (e.g., `rider_distance_km`) have different relationships with the outcome in regime 2
- Summary statistics (mean = 18.4) are meaningless — they describe neither peak accurately

## **Modeling Mistake If Ignored**

The model will learn a **spurious mid-range decision boundary** — treating orders in the 18–25 minute range as ambiguous when in reality they belong cleanly to one of two regimes. This produces high error rates for exactly the orders that matter most for the 30-minute SLA. Concretely: the model may predict low late-delivery risk for a rerouted order (regime 2) because it looks like a fast order on individual features, while missing that rerouting itself is the strongest signal of likely lateness.

**Fix:** Add a binary feature `is_rerouted_order` (derivable from hub assignment logs) to explicitly encode which regime each order belongs to.

# **Part (e) — Business Trade-Off: Precision or Recall?**

## **ZeptoFresh should prioritise Recall — with a nuance.**

Kavya's statement defines the two costs explicitly:

| Error | What happens | Cost |
|---|---|---|
| **False Negative** (missed late order) | Order goes late without intervention → customer churns | Lost customer LTV + negative word-of-mouth |
| **False Positive** (flagged but actually on time) | Unnecessary rider reallocation | Operational cost: fuel, dispatcher time, disrupted routing |

**Primary recommendation: maximise Recall.** A missed late delivery (FN) results in a customer who receives a 30+ minute order, experiences the brand promise being broken, and churns — a permanent revenue loss. An unnecessary reallocation (FP) costs money but retains the customer and preserves the brand.

**The nuance from Kavya's exact words:** *"If we aggressively flag too many orders as late risk, operational costs increase."* This means Recall cannot be maximised without bound — a model that flags every order as late risk has 100% Recall and 100% operational waste. The practical target is **high Recall with a Precision floor** — for example, Recall ≥ 0.80 at Precision ≥ 0.50.

**Metric to optimise:** F-beta score with β > 1 (e.g., F2), which weights Recall more heavily than Precision while still penalising extremely low Precision.

In [ ]:

def f_beta(precision, recall, beta):
    return (1 + beta**2) * (precision * recall) / ((beta**2 * precision) + recall)

scenarios = [
    {"label": "High Recall, Low Precision",    "precision": 0.45, "recall": 0.88},
    {"label": "Balanced (F1)",                  "precision": 0.70, "recall": 0.70},
    {"label": "High Precision, Low Recall",     "precision": 0.90, "recall": 0.45},
]

rows = []
for s in scenarios:
    rows.append({
        "Scenario":  s["label"],
        "Precision": s["precision"],
        "Recall":    s["recall"],
        "F1":        round(f_beta(s["precision"], s["recall"], beta=1), 3),
        "F2 (β=2)": round(f_beta(s["precision"], s["recall"], beta=2), 3)
    })

pd.DataFrame(rows)


                         Scenario  Precision  Recall     F1  F2 (β=2)
0  High Recall, Low Precision        0.45     0.88  0.595     0.746
1  Balanced (F1)                     0.70     0.70  0.700     0.700
2  High Precision, Low Recall        0.90     0.45  0.600     0.497

F2 score correctly ranks the High Recall scenario highest (0.746) for ZeptoFresh — matching the business priority of catching late deliveries even at the cost of extra interventions. F1 would incorrectly favour the Balanced scenario.

# **Part (f) — Advanced Feature Engineering**

## **Three Engineered Features**

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "order_id":           [1001, 1002, 1003, 1004, 1005],
    "prep_time_mins":     [6, 12, 3, 18, 8],
    "rider_distance_km":  [1.2, 3.8, 0.9, 5.1, 2.3],
    "delivery_time_mins": [14, 31, 11, 42, 19],
    "order_hour":         [13, 20, 9, 19, 14],
    "items_count":        [3, 8, 2, 12, 5],
    "order_value_Rs":     [280, 850, 150, 1400, 420],
    "rain_flag":          [0, 1, 0, 1, 0],
    "is_weekend":         [0, 1, 0, 0, 1]
})

# Feature 1: total_fulfillment_time = prep_time_mins + (rider_distance_km * estimated_ride_min_per_km)
# Rationale: delivery_time_mins already exists as the label; this separates the two components
# that drive it — how long the hub takes vs. how far the rider must travel.
# A high prep_time with a short distance signals a hub bottleneck.
# A low prep_time with a long distance signals a routing problem.
AVG_RIDER_SPEED_KM_PER_MIN = 0.5 
df["estimated_ride_time"] = (df["rider_distance_km"] / AVG_RIDER_SPEED_KM_PER_MIN).round(2)
df["total_fulfillment_time"] = df["prep_time_mins"] + df["estimated_ride_time"]

# Feature 2: is_peak_hour = 1 if order_hour in {12–14, 19–21}, else 0
# Rationale: hub congestion and rider unavailability are highest during lunch and dinner rush.
# This directly encodes the operational cause of the second bimodal peak.
peak_hours = set(range(12, 15)) | set(range(19, 22))
df["is_peak_hour"] = df["order_hour"].apply(lambda h: 1 if h in peak_hours else 0)

# Feature 3: complexity_score = items_count * log1p(order_value_Rs)
# Rationale: large, high-value orders require more picking, packing, and verification time.
# Neither items_count nor order_value_Rs alone captures this — a 12-item ₹150 order
# (cheap bulk) behaves differently from a 3-item ₹1,400 order (premium few items).
# The product captures both dimensions simultaneously.
df["complexity_score"] = (df["items_count"] * np.log1p(df["order_value_Rs"])).round(2)

df[[
    "order_id", "prep_time_mins", "rider_distance_km",
    "total_fulfillment_time", "order_hour", "is_peak_hour",
    "items_count", "order_value_Rs", "complexity_score"
]]


   order_id  prep_time_mins  rider_distance_km  total_fulfillment_time  order_hour  is_peak_hour  items_count  order_value_Rs  complexity_score
0      1001               6                1.2                    8.40          13             1            3             280             17.03
1      1002              12                3.8                   19.60          20             1            8             850             53.93
2      1003               3                0.9                    4.80           9             0            2             150             10.12
3      1004              18                5.1                   28.20          19             1           12            1400             87.91
4      1005               8                2.3                   12.60          14             1            5             420             30.46


**Feature summary:**

| Feature | Formula | Why it helps |
|---|---|---|
| `total_fulfillment_time` | `prep_time_mins + (rider_distance_km / avg_speed)` | Decomposes delivery time into hub bottleneck vs. routing problem — each has a different fix |
| `is_peak_hour` | `1 if order_hour ∈ {12–14, 19–21} else 0` | Directly encodes the operational cause of the bimodal second peak observed in Tier-1 cities |
| `complexity_score` | `items_count × log1p(order_value_Rs)` | Captures orders that are simultaneously large and high-value — the strongest predictor of prep time overruns |